## Related Documents Transformer (with bonus IIIF ID transformer!)

This script takes a JBPP scanning log page, in its standardized form, and converts it into two exports that can be fed into the Drupal database. It is a bit sensitive, though -- recommend at least some facility with Python. Maybe ought to try and make it an Excel template?

The two base functions that make the related documents all work are AI-generated, with some fine-tuning on my part.

This exports three different CSV files, since we like to try and compartmentalize things with the feeds. But the image ID processing could easily be combined with the related document processing.

A note: any related documents present in Drupal but not in the Scanning Log **will be overwritten**. This should thus be done early in the document workflow to minimize overwrites.

In [1]:
import pandas as pd
import numpy as np
import re

In [3]:
filepath = 'scanlog_b18_f3_f10.csv'
log = pd.read_csv(filepath)

In [5]:
log.columns

Index(['Tracksys Information', 'FOLDER NUMBER', 'Who Cataloged?',
       'DOCUMENT ID', 'TITLE', 'Document Length',
       'INTERNAL NOTES "Bulk imported" is automatically entered. Add any other notes after it.',
       'Manuscript Type (for Catalog Creation Only) Autograph | Copy | Draft | Fragment | Handwritten | Letterhead | Printed | Signed | Typed | Version',
       'Related Documents (for Catalog Creation Only) Choose General, Enclosure, or Response ',
       'TIFFs Created', 'PDF or IIIF manifest Created', 'UPLOADED TO F.T.P',
       'UVA IIIF IDs', 'UVA IIIF Imported into Drupal',
       'Editor Moved to Storage', 'Notes'],
      dtype='object')

## Expanding

I want to try and make this something that I can do for an entire box at once, hopefully. I still understand the importance of processing each folder one at a time, but I want to create the CSVs for each folder all at once. Let's see if I still got it fr

In [7]:
log = log[['FOLDER NUMBER', 'DOCUMENT ID', 'TITLE', 'Related Documents (for Catalog Creation Only) Choose General, Enclosure, or Response ', 'UVA IIIF IDs']]
log.columns = ['Folder', 'ID', 'Title', 'Related', 'Image Identifier']
log

,Folder,ID,Title,Related,Image Identifier
0,Box 18 Folder 3,PJB 9456,"From Julian Bond to Hugh Cline, 3 Jun 1974",NaN,3442947
1,Box 18 Folder 3,PJB 9457,"To Julian Bond from Hugh Cline, 20 May 1974, w...","Response: From Julian Bond to Hugh Cline, 3 Ju...",3442948
2,Box 18 Folder 3,PJB 9458,Draft of 3 separate letters: From Julian Bond ...,NaN,3442949
3,Box 18 Folder 3,PJB 9459,"To Julian Bond from Dr. Helen Wise, 3 Jun 1974",Response: Draft of 3 separate letters: From Ju...,3442950
4,Box 18 Folder 3,PJB 9460,To Julian Bond from Mrs. B. Edwards and Mrs. J...,Response: Draft of 3 separate letters: From Ju...,3442951
...,...,...,...,...,...
1462,Box 18 Folder 10,PJB 10924,From Betty Carter for Julian Bond to Marian Ch...,NaN,3447391
1463,Box 18 Folder 10,PJB 10925,From Betty Carter for Julian Bond to Sidney Sc...,NaN,3447392
1464,NaN,NaN,NaN,NaN,NaN
1465,NaN,NaN,NaN,NaN,NaN


In [28]:
temp = log[log['ID'].notna()]
filtered = temp[temp['ID'].str.startswith('PJB')]

This should be enough in terms of filtering. We shall see if we encounter any edge cases.

In [33]:
filtered.ID = filtered.ID.apply(lambda x: x.strip('PJB '))

C:\Users\charl\AppData\Local\Temp\ipykernel_16216\3037750040.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered.ID = filtered.ID.apply(lambda x: x.strip('PJB '))


In [37]:
filtered.head()

,Folder,ID,Title,Related,Image Identifier
0,Box 18 Folder 3,9456,"From Julian Bond to Hugh Cline, 3 Jun 1974",NaN,3442947
1,Box 18 Folder 3,9457,"To Julian Bond from Hugh Cline, 20 May 1974, w...","Response: From Julian Bond to Hugh Cline, 3 Ju...",3442948
2,Box 18 Folder 3,9458,Draft of 3 separate letters: From Julian Bond ...,NaN,3442949
3,Box 18 Folder 3,9459,"To Julian Bond from Dr. Helen Wise, 3 Jun 1974",Response: Draft of 3 separate letters: From Ju...,3442950
4,Box 18 Folder 3,9460,To Julian Bond from Mrs. B. Edwards and Mrs. J...,Response: Draft of 3 separate letters: From Ju...,3442951


In [39]:
def multi_replace(text, replace_map):
    """
    Replace multiple substrings in `text` based on a mapping dict.
    - Keys are replaced with their values
    - Longest keys are prioritized
    - Special regex characters in keys are escaped
    """
    if not isinstance(text, str):
        return text  # return unchanged if not a string
    
    replace_map = {str(k): str(v) for k, v in replace_map.items()}
    
    # Sort keys by length (longest first)
    sorted_keys = sorted(replace_map.keys(), key=len, reverse=True)
    
    # Build regex pattern with escaped keys
    pattern = re.compile("|".join(re.escape(k) for k in sorted_keys))
    
    # Replace using a lambda that looks up the match
    return pattern.sub(lambda m: replace_map[m.group(0)], text)

In [41]:
# Thanks ChatGPT

def extract_labels(text):
    labels = ["General", "Enclosure", "Response"]
    out = {lbl: None for lbl in labels}
    out["unmatched"] = False   # flag
    
    if not isinstance(text, str):
        return out
    
    consumed_spans = []  # track matched spans
    
    for label in labels:
        # find all occurrences like "General: 1, 2"
        for m in re.finditer(rf"{label}:\s*([\d, ]+)", text):
            consumed_spans.append(m.span())
            numbers = [n.strip() for n in m.group(1).split(",") if n.strip()]
            
            if out[label]:
                out[label] += "|" + "|".join(numbers)
            else:
                out[label] = "|".join(numbers)
    
    # Mark unmatched text if there is leftover after removing matches
    if consumed_spans:
        mask = [" "] * len(text)
        for start, end in consumed_spans:
            mask[start:end] = "_" * (end-start)  # mark matched regions
        leftover = "".join(ch for ch, m in zip(text, mask) if m == " ")
        if leftover.strip():   # anything left that isn’t just spaces/commas
            out["unmatched"] = True
    else:
        # Nothing matched at all → flag whole text as unmatched
        if text.strip():
            out["unmatched"] = True
    
    return out

In [43]:
# This may be a rather hefty set of things that needs doing here

In [45]:
map_file = 'node_field_data.csv'
mapping_df = pd.read_csv(map_file)
mapping_df.shape

(2870, 20)

In [47]:
mapping_df.columns

Index(['nid', 'vid', 'type', 'langcode', 'status', 'uid', 'title', 'created',
       'changed', 'promote', 'sticky', 'default_langcode',
       'revision_translation_affected', 'gva_node_layout', 'gva_breadcrumb',
       'gva_header', 'gva_node_class', 'gva_box_layout',
       'gva_pagebuilder_content', 'gva_pagebuilder_enable'],
      dtype='object')

In [49]:
replace_map = dict(zip(mapping_df.title, mapping_df.nid))

In [51]:
# this should set up our replace_map correctly. Now we get into sorting out the individual folders and our bigger helper functions here

In [57]:
# For folder sort:

folders = filtered.groupby('Folder')

In [93]:
def sort_related_docs(folder_log, replace_map, folder_name):
    """
    This helper function takes a dataframe with a single folder's worth of documents and sorts the related documents appropriately.

    Exports to exports folder for human review and provides a basic summary of the function's success or failure.

    So still requires human-in-the-loop and all.
    """
    # run the related docs through matching: this just helps verify that every doc title matches an actual doc in Drupal
    folder_log['Related'] = folder_log['Related'].apply(lambda x: multi_replace(x, replace_map))
    related_df = folder_log[['ID', 'Related']]

    # if "related" is empty, separate it out and make a note
    related_export = related_df[related_df['Related'].notna()]
    no_nulls = len(related_df) - len(related_export)
    print(f'Total number of rows in {folder_name}: {len(related_df)}.')
    print(f'Dropping {no_nulls} empty rows.')

    if related_export.empty:
        print(f'{folder_name} has no related document metadata. Skipping {folder_name}.')
        return None


    # label
    expanded = related_export['Related'].apply(extract_labels).apply(pd.Series)
    related_export = pd.concat([related_export, expanded], axis=1)

    # sort out the unmatched documents, leaving just the ones ready to go
    related_feed = related_export[related_export['unmatched'] == False].drop(['unmatched', 'Related'], axis = 1)
    related_feed.to_csv(f'Exports/related_docs_{folder_name}_feed.csv', index = False)

    # keep 'Related' so we can see what exactly the problem was with the stuff that needs to be manually checked
    needs_manual_entry = related_export[related_export['unmatched'] == True].drop('unmatched', axis = 1)
    needs_manual_entry.to_csv(f'Exports/needs_manual_entry_{folder_name}.csv', index = False)

    print(f'Related Documents for {folder_name} exported successfully. Number of unmatched documents: {len(needs_manual_entry)}.')

    return None

In [81]:
def clean_image_ids(folder_log, folder_name):
    # very short and sweet
    image_export = folder_log.drop(['Folder', 'Title', 'Related'], axis = 1)
    image_export.to_csv(f'Exports/image_id_feed_{folder_name}.csv', index = False)
    print(f'Image Identifiers for {folder_name} exported successfully.')
    return None

In [99]:
# let er rip

for folder_name, folder_log in folders:
    sort_related_docs(folder_log, replace_map, folder_name)
    clean_image_ids(folder_log, folder_name)
    print('\n')

Total number of rows in Box 18 Folder 10: 372.
Dropping 358 empty rows.
Related Documents for Box 18 Folder 10 exported successfully. Number of unmatched documents: 0.
Image Identifiers for Box 18 Folder 10 exported successfully.


Total number of rows in Box 18 Folder 3: 197.
Dropping 160 empty rows.
Related Documents for Box 18 Folder 3 exported successfully. Number of unmatched documents: 3.
Image Identifiers for Box 18 Folder 3 exported successfully.


Total number of rows in Box 18 Folder 4: 10.
Dropping 10 empty rows.
Box 18 Folder 4 has no related document metadata. Skipping Box 18 Folder 4.
Image Identifiers for Box 18 Folder 4 exported successfully.


Total number of rows in Box 18 Folder 5: 1.
Dropping 0 empty rows.
Related Documents for Box 18 Folder 5 exported successfully. Number of unmatched documents: 0.
Image Identifiers for Box 18 Folder 5 exported successfully.


Total number of rows in Box 18 Folder 6: 234.
Dropping 173 empty rows.
Related Documents for Box 18 Folder

In [85]:
# reckon that works! Can see still definitely requires some human checking, but quite happy with that.